In [ ]:
from pathlib import Path
import pandas as pd
from london_cycles import stations as load_stations


# Ścieżki do danych
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)

    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path

    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"
DATA_FILE = DATA_DIR / "rides_data_without_discontinuities.parquet"


# Wczytanie danych o wypożyczeniach
rides_data = pd.read_parquet(DATA_FILE, columns=[
    "start_station_id",
    "start_station_name",
    "start_date",
    "start_day_of_week"
])

# Wczytanie historycznych danych stacji z Hugging Face
stations_df = pd.DataFrame(load_stations())


# Analizowany okres: 2024–2025
start = pd.Timestamp("2024-01-01")
end = pd.Timestamp("2026-01-01")

# Kopia danych historycznych
stations = stations_df.copy()

# Konwersja dat
for col in ["valid_from", "valid_to", "install_date"]:
    stations[col] = pd.to_datetime(stations[col], errors="coerce")

# Ujednolicenie identyfikatorów
stations["terminal_id"] = pd.to_numeric(
    stations["terminal_name"], errors="coerce"
)

# Zachowanie rekordów stacji zainstalowanych przed 2024 rokiem
stations = stations[
    (stations["installed"] == True) &
    (stations["install_date"].isna() | (stations["install_date"] <= start))
].copy()

# Brak końca ważności oznacza otwarty przedział
stations["valid_to"] = stations["valid_to"].fillna(end)


# Sprawdzenie ciągłości historycznych rekordów
def active_full_period(group):
    covered_until = start

    for row in group.sort_values("valid_from").itertuples():

        if pd.isna(row.valid_from) or row.valid_from > covered_until:
            return False

        covered_until = max(covered_until, row.valid_to)

        if covered_until >= end:
            return True

    return False


# Identyfikatory stacji spełniających warunek
active_station_ids = {
    int(station_id)
    for station_id, group in stations.groupby("terminal_id")
    if active_full_period(group)
}

# Identyfikatory stacji z naszych wypożyczeń
trip_ids = set(
    pd.to_numeric(
        rides_data["start_station_id"], errors="coerce"
    ).dropna().astype(int)
)

# Porównanie zbiorów
valid_stations = trip_ids & active_station_ids
excluded_stations = trip_ids - active_station_ids


# Podsumowanie
print("ANALIZA STACJI: 2024-2025")
print("-" * 40)
print(f"Stacje w wypożyczeniach: {len(trip_ids)}")
print(f"Stacje z ciągłym statusem instalacji: {len(active_station_ids)}")
print(f"Stacje zachowane: {len(valid_stations)}")
print(f"Stacje odrzucone: {len(excluded_stations)}")
print(f"Procent zachowanych stacji: {len(valid_stations) / len(trip_ids) * 100:.2f}%")


# Lista odrzuconych stacji
excluded_summary = (
    rides_data[
        pd.to_numeric(
            rides_data["start_station_id"], errors="coerce"
        ).isin(excluded_stations)
    ]
    .groupby(["start_station_id", "start_station_name"])
    .agg(
        rental_count=("start_date", "size")
    )
    .reset_index()
    .sort_values("rental_count", ascending=False)
)

display(excluded_summary)

ANALIZA STACJI: 2024–2025
----------------------------------------
Stacje w wypożyczeniach: 801
Stacje z ciągłym statusem instalacji: 804
Stacje zachowane: 796
Stacje odrzucone: 5
Procent zachowanych stacji: 99.38%


,start_station_id,start_station_name,rental_count
2,300060,"Brandon Street, Walworth",18180
1,200032,"Kennington Oval, Oval",16833
3,300074,"Newton Street, Covent Garden",14570
0,22181,"Camley Street, Camden Town",7644
4,300099,"Adeline Place, Bloomsbury",832


In [12]:
# Historyczne informacje o odrzuconych stacjach
excluded_details = stations_df.copy()

# Ujednolicenie identyfikatorów
excluded_details["terminal_id"] = pd.to_numeric(
    excluded_details["terminal_name"], errors="coerce"
)

# Wybór odrzuconych stacji
excluded_details = excluded_details[
    excluded_details["terminal_id"].isin(excluded_stations)
].copy()

# Wyświetlenie historii stacji
display(
    excluded_details[
        [
            "terminal_id",
            "common_name",
            "installed",
            "install_date",
            "removal_date",
            "valid_from",
            "valid_to"
        ]
    ].sort_values(["terminal_id", "valid_from"])
)

,terminal_id,common_name,installed,install_date,removal_date,valid_from,valid_to
867,22181.0,"Camley Street, Camden Town",True,2011-07-07,NaN,2024-06-12,2024-06-13
868,22181.0,"Camley Street, Camden Town",True,2011-07-07,NaN,2024-06-13,NaN
348,200032.0,"Kennington Oval, Oval",True,2012-01-04,NaN,2022-04-29,2024-05-21
864,200032.0,"Kennington Oval, Oval",False,NaN,NaN,2024-05-20,NaN
866,200032.0,"Kennington Oval, Oval",True,2012-01-04,NaN,2024-06-03,NaN
800,300060.0,"Brandon Street, Walworth",True,NaN,NaN,2022-08-30,2022-09-11
806,300060.0,"Brandon Street, Walworth",True,NaN,NaN,2022-09-12,2025-01-07
884,300060.0,"Brandon Street, Walworth",True,NaN,NaN,2025-01-09,2025-01-10
886,300060.0,"Brandon Street, Walworth",True,NaN,NaN,2025-01-11,2025-01-15
888,300060.0,"Brandon Street, Walworth",True,NaN,NaN,2025-01-17,NaN


In [13]:
# Odrzucone stacje
ids_to_check = [22181, 200032, 300060, 300074, 300099]

# Kopia danych
station_history = stations_df.copy()

# Identyfikatory terminali
station_history["terminal_id"] = pd.to_numeric(
    station_history["terminal_name"], errors="coerce"
)

# Historia odrzuconych stacji
display(
    station_history[
        station_history["terminal_id"].isin(ids_to_check)
    ][
        [
            "terminal_id",
            "place_id",
            "common_name",
            "lat",
            "lon",
            "installed",
            "valid_from",
            "valid_to"
        ]
    ].sort_values(["terminal_id", "valid_from"])
)

,terminal_id,place_id,common_name,lat,lon,installed,valid_from,valid_to
867,22181.0,BikePoints_425,"Camley Street, Camden Town",51.537471,-0.130294,True,2024-06-12,2024-06-13
868,22181.0,BikePoints_425,"Camley Street, Camden Town",51.537920,-0.130529,True,2024-06-13,NaN
348,200032.0,BikePoints_440,"Kennington Oval, Oval",51.483145,-0.113134,True,2022-04-29,2024-05-21
864,200032.0,BikePoints_860,"Kennington Oval, Oval",51.483145,-0.113134,False,2024-05-20,NaN
866,200032.0,BikePoints_440,"Kennington Oval, Oval",51.483145,-0.113134,True,2024-06-03,NaN
800,300060.0,BikePoints_850,"Brandon Street, Walworth",51.489102,-0.091548,True,2022-08-30,2022-09-11
806,300060.0,BikePoints_850,"Brandon Street, Walworth",51.489102,-0.091548,True,2022-09-12,2025-01-07
884,300060.0,BikePoints_850,"Brandon Street, Walworth",51.489102,-0.091548,True,2025-01-09,2025-01-10
886,300060.0,BikePoints_850,"Brandon Street, Walworth",51.489102,-0.091548,True,2025-01-11,2025-01-15
888,300060.0,BikePoints_850,"Brandon Street, Walworth",51.489102,-0.091548,True,2025-01-17,NaN


In [14]:
# Przygotowanie danych
rides_test = rides_data.copy()

rides_test["start_date"] = pd.to_datetime(rides_test["start_date"])
rides_test["start_station_id"] = pd.to_numeric(
    rides_test["start_station_id"], errors="coerce"
)

# Okresy do sprawdzenia
periods_to_check = [
    ("Camley Street", 22181, "2024-01-01", "2024-06-12"),
    ("Kennington Oval", 200032, "2024-05-21", "2024-06-03"),
    ("Brandon Street", 300060, "2025-01-07", "2025-01-17"),
    ("Newton Street", 300074, "2025-04-11", "2025-05-08"),
    ("Abyssinia Close", 300099, "2024-01-01", "2025-10-29"),
    ("Adeline Place", 300099, "2025-10-29", "2026-01-01")
]

# Sprawdzenie wypożyczeń
results = []

for name, station_id, start_date, end_date in periods_to_check:

    subset = rides_test[
        (rides_test["start_station_id"] == station_id) &
        (rides_test["start_date"] >= start_date) &
        (rides_test["start_date"] < end_date)
    ]

    results.append({
        "station": name,
        "period": f"{start_date} – {end_date}",
        "rental_count": len(subset),
        "station_names": ", ".join(
            subset["start_station_name"].dropna().unique()
        )
    })

display(pd.DataFrame(results))

,station,period,rental_count,station_names
0,Camley Street,2024-01-01 – 2024-06-12,1,"Camley Street, Camden Town"
1,Kennington Oval,2024-05-21 – 2024-06-03,0,
2,Brandon Street,2025-01-07 – 2025-01-17,13,"Brandon Street, Walworth"
3,Newton Street,2025-04-11 – 2025-05-08,327,"Newton Street, Covent Garden"
4,Abyssinia Close,2024-01-01 – 2025-10-29,0,
5,Adeline Place,2025-10-29 – 2026-01-01,832,"Adeline Place, Bloomsbury"


In [15]:
# Stacje i okresy do sprawdzenia
periods_to_check = {
    "Brandon Street": (300060, "2025-01-07", "2025-01-17"),
    "Newton Street": (300074, "2025-04-11", "2025-05-08"),
    "Kennington Oval": (200032, "2024-05-20", "2024-06-03")
}

# Dzienne wypożyczenia w podejrzanych okresach
for name, (station_id, start, end) in periods_to_check.items():

    subset = rides_test[
        (rides_test["start_station_id"] == station_id) &
        (rides_test["start_date"] >= start) &
        (rides_test["start_date"] < end)
    ].copy()

    daily_counts = (
        subset.groupby(subset["start_date"].dt.date)
        .size()
    )

    print(f"\n{name}")
    print(daily_counts.to_string())


Brandon Street
start_date
2025-01-16    13

Newton Street
start_date
2025-04-11     5
2025-04-12     3
2025-04-13     7
2025-04-14    11
2025-04-15    12
2025-04-16     6
2025-04-17    10
2025-04-18     5
2025-04-19     4
2025-04-20     6
2025-04-21     3
2025-04-22    12
2025-04-23    28
2025-04-24    15
2025-04-25    16
2025-04-26    11
2025-04-27    13
2025-04-28    13
2025-04-29    24
2025-04-30    26
2025-05-01    17
2025-05-02    23
2025-05-03     5
2025-05-04     5
2025-05-05     2
2025-05-06    16
2025-05-07    29

Kennington Oval
Series([], )


In [16]:
# Okresy do sprawdzenia
periods = {
    "Kennington Oval": (200032, "2024-05-10", "2024-06-15"),
    "Camley Street": (22181, "2024-01-01", "2024-06-20")
}

for name, (station_id, start, end) in periods.items():

    subset = rides_test[
        (rides_test["start_station_id"] == station_id) &
        (rides_test["start_date"] >= start) &
        (rides_test["start_date"] < end)
    ]

    daily_counts = (
        subset.groupby(subset["start_date"].dt.normalize())
        .size()
    )

    # Uwzględnienie dni bez wypożyczeń
    days = pd.date_range(start, end, inclusive="left")
    daily_counts = daily_counts.reindex(days, fill_value=0)

    print(f"\n{name}")
    print(f"Dni z wypożyczeniami: {(daily_counts > 0).sum()}")
    print(f"Dni bez wypożyczeń: {(daily_counts == 0).sum()}")

    display(daily_counts.to_frame("rental_count").T)


Kennington Oval
Dni z wypożyczeniami: 22
Dni bez wypożyczeń: 14


,2024-05-10,2024-05-11,2024-05-12,2024-05-13,2024-05-14,2024-05-15,2024-05-16,2024-05-17,2024-05-18,2024-05-19,...,2024-06-05,2024-06-06,2024-06-07,2024-06-08,2024-06-09,2024-06-10,2024-06-11,2024-06-12,2024-06-13,2024-06-14
rental_count,42,55,38,24,22,32,18,38,37,1,...,39,30,50,18,20,18,30,21,20,30



Camley Street
Dni z wypożyczeniami: 9
Dni bez wypożyczeń: 162


,2024-01-01,2024-01-02,2024-01-03,2024-01-04,2024-01-05,2024-01-06,2024-01-07,2024-01-08,2024-01-09,2024-01-10,...,2024-06-10,2024-06-11,2024-06-12,2024-06-13,2024-06-14,2024-06-15,2024-06-16,2024-06-17,2024-06-18,2024-06-19
rental_count,0,0,0,0,0,0,0,0,0,0,...,0,1,2,4,6,4,9,7,9,8
